# Dynamic Partition Pruning (DPP)
- Pruning partitions at runtime
- Problem Statement: Analyse the listening activity of users on the release date of a song on/after `2020-01-01`

**What this notebook does**

* Writes the listening activity data as parquet **partitioned by `listen_date`** (one folder per day).
* Filters songs released on/after 2020-01-01 and joins them to the listening activity on `listen_date = release_date` and `song_id`.
* Inspects the physical plan to see Spark prune `listen_date` partitions **at runtime** using the result of the songs filter.

**Core concept: static vs dynamic partition pruning**

* **Static pruning:** a filter written directly on the partition column (`WHERE listen_date = '2023-07-18'`) is known when the query is planned, so Spark lists only the matching folders.
* **Dynamic pruning:** here the filter is on the *other* table (`release_date > 2019-12-31` on songs). Spark does not know which `listen_date` values matter until it has evaluated that filter. With DPP, Spark first computes the filtered songs, collects their `release_date` values, and injects them as a partition filter on the listening-activity scan, so folders for non-matching dates are never read.

**Requirements for DPP:** the large (fact) table must be partitioned on a join key, the other (dimension) side must have a filter, and the join must be an equi-join. It is most efficient when the dimension side is small enough to broadcast, because the broadcast result is reused as the pruning filter. It is enabled by default (`spark.sql.optimizer.dynamicPartitionPruning.enabled`).

**Databricks conversion notes**

* The local `SparkSession.builder` / `sparkContext` setup was removed: Databricks provides `spark`.
* Raw CSVs are read from `/Volumes/lingaro_sandbox_ne/sandani_sandbox/volume_managed/data/input/`; the partitioned parquet output is written to `/Volumes/lingaro_sandbox_ne/sandani_sandbox/volume_managed/data/output/listening_activity_pt_dpp` (a new folder, so existing `listening_activity_pt*` outputs are untouched).
* CSV schema inference on Databricks reads `listen_date` / `release_date` as **timestamp**; `to_date` converts them to dates exactly as before.
* The join condition now references `df_selected_songs` (the filtered DataFrame) instead of `df_songs`; under Spark Connect, mixing column references from a parent DataFrame can fail to resolve.
* Recommended compute: an interactive cluster such as *Sandani Personal Compute*.

## 1. Setup

In [0]:
import warnings
warnings.filterwarnings("ignore")

In [0]:
# Jupyter-only InteractiveShell setting removed; Databricks renders cell output natively.

# Unity Catalog volume locations (replace the local ../data/partitioning paths)
input_path = "/Volumes/lingaro_sandbox_ne/sandani_sandbox/volume_managed/data/input"
# Separate output folder so the existing data/output/listening_activity_pt* folders are not overwritten
partitioned_path = "/Volumes/lingaro_sandbox_ne/sandani_sandbox/volume_managed/data/output/listening_activity_pt_dpp"

In [0]:
from pyspark.storagelevel import StorageLevel
from pyspark.sql.types import *
import pyspark.sql.functions as F
# SparkSession is not imported: Databricks provides a ready-to-use `spark` object.

In [0]:
# Original (local Jupyter):
#   spark = SparkSession.builder.config("spark.driver.memory", "10g").master("local[*]")
#           .appName("6_1_dynamic_partition_pruning").getOrCreate()
#   sc = spark.sparkContext; sc.setLogLevel("ERROR")
# On Databricks the session already exists; driver memory is set by the cluster.

print(f"Spark version: {spark.version}")
print("dynamicPartitionPruning.enabled =", spark.conf.get("spark.sql.optimizer.dynamicPartitionPruning.enabled"))

## 2. Partition the Listening Activity (Fact Table)

**What:** Read the raw CSV, rename the original timestamp column to `listen_time`, derive a `listen_date` (date only), and write parquet with `partitionBy("listen_date")`.

**Core concept:** `partitionBy` creates one sub-folder per value, e.g. `listening_activity_pt/listen_date=2023-07-18/part-....parquet`. The partition value is stored in the folder name, not in the files, and is restored as a column when the folder is read back. Any filter on `listen_date`, static or dynamic, can then skip whole folders without opening them. `mode("overwrite")` replaces the output folder on every run.

In [0]:
df_listening_actv = spark.read.csv(f"{input_path}/Spotify_Listening_Activity.csv", header=True, inferSchema=True)
df_listening_actv = (
    df_listening_actv
    .withColumnRenamed("listen_date", "listen_time")
    .withColumn("listen_date", F.to_date("listen_time", "yyyy-MM-dd HH:mm:ss.SSSSSS"))
)

# Partitioning listening activity by the listen date
(
    df_listening_actv
    .write
    .partitionBy("listen_date")
    .mode("overwrite")
    .parquet(partitioned_path)
)

In [0]:
df_listening_actv_pt = spark.read.parquet(partitioned_path)
display(df_listening_actv_pt.limit(5))

## 3. Prepare the Songs (Dimension Table)

**What:** Read the songs CSV and derive a date-only `release_date` from the original timestamp, so it can be compared with the `listen_date` partition column.

**Core concept:** DPP can only prune when the join key on the dimension side has the **same type and granularity** as the partition column. Joining a timestamp to a date would never match exactly and would prevent pruning, which is why both sides are converted with `to_date`.

In [0]:
df_songs = spark.read.csv(f"{input_path}/Spotify_Songs.csv", header=True, inferSchema=True)
df_songs.printSchema()

In [0]:
df_songs = (
    df_songs
    .withColumnRenamed("release_date", "release_datetime")
    .withColumn("release_date", F.to_date("release_datetime", "yyyy-MM-dd HH:mm:ss.SSSSSS"))
)
display(df_songs.limit(5))
df_songs.printSchema()

## 4. Join and Observe Dynamic Partition Pruning

**What:** Keep songs released after 2019-12-31, join them to the partitioned listening activity on `listen_date = release_date` and `song_id`, and print the plan with `explain(True)`.

**Core concept:** The filter is only on songs, yet the join key `release_date` maps to the partition column `listen_date`. Spark therefore adds a runtime subquery that collects the qualifying `release_date` values and applies them to the fact table scan.

**What to look for in the physical plan:**

* On the listening-activity `FileScan parquet`: `PartitionFilters: [isnotnull(listen_date), dynamicpruning#...]`. This is DPP. With AQE (on by default) `explain()` prints the *initial* plan; after the query runs, the final plan in the Spark UI shows it as `dynamicpruningexpression(listen_date IN subquery...)`.
* A `BroadcastHashJoin` with the filtered songs broadcast (`Exchange SinglePartition, EXECUTOR_BROADCAST`); the same broadcast result is reused to build the pruning filter.

**Where to see the effect:** in the Spark UI > SQL / DataFrame tab, open the query and check the scan node of the listening activity: *number of partitions read* is lower than the total number of `listen_date` folders.

In [0]:
# Pick songs released in 2020
df_selected_songs = df_songs.filter(F.col("release_date") > F.lit("2019-12-31"))


df_listening_actv_of_selected_songs = df_listening_actv_pt.join(
    df_selected_songs, 
    # Columns referenced from df_selected_songs (original used df_songs) so they resolve under Spark Connect
    on=(df_selected_songs.release_date == df_listening_actv_pt.listen_date) & (df_selected_songs.song_id == df_listening_actv_pt.song_id),
    how="inner"
)

In [0]:
df_listening_actv_of_selected_songs.explain(True)

In [0]:
# show() is used instead of display() because the result has two song_id columns
df_listening_actv_of_selected_songs.show()

## Summary

* The fact table is partitioned by `listen_date` (126 date folders in this dataset).
* The filter is only on the songs table (78 songs released on/after 2020-01-01), but because the join key maps to the partition column, Spark builds a runtime partition filter from those release dates.
* Only the folders whose `listen_date` equals one of the selected release dates need to be read (3 dates match in this data), instead of all 126.

**Static vs dynamic pruning:** static pruning needs the filter value at planning time; dynamic pruning derives it at runtime from another table. Both depend on the fact table being partitioned on the filtered/join column.

**On Delta tables:** the same idea applies to Delta tables in Unity Catalog. Databricks additionally uses file-level statistics (data skipping, liquid clustering) to prune files dynamically even when a table is not partitioned.